# Pump Scenario Sample Generator & Visualiser

Runs all **10 pump scenarios** (1 normal + 3 failures + 6 decoys) serially using NASA's **progpy CentrifugalPump** model with:
- Cycling voltage load (1h period) for natural signal variation
- Per-signal `process_noise` and `measurement_noise` for realistic sensor jitter
- Wear rates set via `x0` initial state (`wA=0.01` → failure ~6h)

| Group | Failure mode | Discriminating signals | Hard negatives |
|-------|-------------|----------------------|----------------|
| Bearing | `wA=0.01` | shaft speed ↓ **AND** temperature ↑ | setpoint step/ramp (speed ↓, temp flat) |
| Impeller | `wA=0.01` | flow_out ↓ **AND** Qin/Qout ratio > 1 | back-pressure step/ramp (ratio ≈ 1) |
| Seal | `wA=0.01` | flow_out ↓ **AND** flow_in stable (gap grows) | flow-reduction step/ramp (gap ≈ 0) |

In [1]:
import sys, importlib, warnings
warnings.filterwarnings('ignore')

# Make sample_generator importable and reload if already loaded
if "sample_generator" in sys.modules:
    importlib.reload(sys.modules["sample_generator"])
from sample_generator import build_scenarios, run_all

## Run all 9 simulations
Each scenario runs ~360 steps (6 h at 1 sample/min). Faulty scenarios stop early at the failure threshold.

In [2]:
scenarios = build_scenarios()
results = run_all(scenarios)

  Simulating: normal                             

 361 rows  [{'NORMAL': 361}]
  Simulating: pump_bearing_wear                  

 355 rows  [{'NORMAL': 294, 'PRE_FAILURE': 60, 'FAILURE': 1}]
  Simulating: decoy_highload_step                

 361 rows  [{'NORMAL': 361}]
  Simulating: decoy_highload_ramp                

 361 rows  [{'NORMAL': 361}]
  Simulating: pump_impeller_wear                 

...

 361 rows  [{'NORMAL': 361}]
  Simulating: decoy_back_pressure_ramp           

 361 rows  [{'NORMAL': 361}]
  Simulating: pump_radial_wear                   

 361 rows  [{'NORMAL': 300, 'PRE_FAILURE': 60, 'FAILURE': 1}]
  Simulating: decoy_radial_highload_step         

 361 rows  [{'NORMAL': 361}]
  Simulating: decoy_radial_highload_ramp         

 361 rows  [{'NORMAL': 361}]


In [3]:
# Quick summary table
import pandas as pd

rows = []
for name, df in results.items():
    vc = df["label"].value_counts()
    rows.append({
        "scenario": name,
        "rows": len(df),
        "NORMAL": vc.get("NORMAL", 0),
        "PRE_FAILURE": vc.get("PRE_FAILURE", 0),
        "FAILURE": vc.get("FAILURE", 0),
    })
pd.DataFrame(rows).set_index("scenario")

,rows,NORMAL,PRE_FAILURE,FAILURE
scenario,,,,
normal,361,361,0,0
pump_bearing_wear,355,294,60,1
decoy_highload_step,361,361,0,0
decoy_highload_ramp,361,361,0,0
pump_impeller_wear,361,300,60,1
decoy_back_pressure_step,361,361,0,0
decoy_back_pressure_ramp,361,361,0,0
pump_radial_wear,361,300,60,1
decoy_radial_highload_step,361,361,0,0


In [4]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sample_generator import COLOURS, LABEL_ALPHA, LABEL_COLOUR

## Plot 0 — Healthy baseline
All signals stable, every row labelled NORMAL. This is what "nothing is wrong" looks like.

In [5]:
df = results["normal"]
pairs = [
    ("shaft_speed_rads", "Shaft speed (rad/s)"),
    ("fluid_temp_K",     "Temperature (K)"),
    ("flow_out_m3s",     "Flow out (m\u00b3/s)"),
    ("flow_in_m3s",      "Flow in (m\u00b3/s)"),
    ("pump_speed_rads",  "Pump speed (rad/s)"),
    ("thrust_bearing_K", "Thrust bearing Tt (K)"),
]

fig = make_subplots(rows=2, cols=3, subplot_titles=[p[1] for p in pairs])
for idx, (col, label) in enumerate(pairs):
    r, c = idx // 3 + 1, idx % 3 + 1
    if col in df.columns:
        fig.add_trace(go.Scatter(x=df["time_h"], y=df[col], mode="lines",
                                 line=dict(color=COLOURS["normal"], width=1.5),
                                 name=label, showlegend=False), row=r, col=c)
        fig.update_xaxes(title_text="Time (hours)", row=r, col=c)

fig.update_layout(title="Healthy pump — baseline signals (all NORMAL)",
                  height=500, template="plotly_white")
fig.show()

## Plot 1 — Thrust bearing wear vs high-load decoys

**Both** show Tt rising — but the failure rises **faster** than the operating conditions explain.

- **Failure:** Tt climbs with no speed increase — excess heat from friction (`rThrust * w^2`)
- **Hard negative:** Higher voltage → speed ↑ → Tt rises too, but proportionally — no excess heat

The discrimination problem: you can't just threshold on "Tt is rising" — you need to check whether the rate of rise is explained by the current load.

In [6]:
def plot_group_interactive(scenarios, signal_y, signal_x, title, y_label, x_label, scatter_label):
    """3-panel interactive plot: signal_y over time, signal_x over time, scatter x vs y."""
    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=[f"{y_label} over time",
                                        f"{x_label} over time",
                                        f"{scatter_label}"])
    for name, label, colour in scenarios:
        df = results[name]
        fig.add_trace(go.Scatter(x=df["time_h"], y=df[signal_y], mode="lines",
                                 line=dict(color=colour, width=1.5),
                                 name=label, legendgroup=label), row=1, col=1)
        fig.add_trace(go.Scatter(x=df["time_h"], y=df[signal_x], mode="lines",
                                 line=dict(color=colour, width=1.5),
                                 name=label, legendgroup=label, showlegend=False), row=1, col=2)
        fig.add_trace(go.Scatter(x=df[signal_x], y=df[signal_y], mode="markers",
                                 marker=dict(color=colour, size=3, opacity=0.5),
                                 name=label, legendgroup=label, showlegend=False), row=1, col=3)
    fig.update_xaxes(title_text="Time (hours)", row=1, col=1)
    fig.update_xaxes(title_text="Time (hours)", row=1, col=2)
    fig.update_xaxes(title_text=x_label, row=1, col=3)
    fig.update_yaxes(title_text=y_label, row=1, col=1)
    fig.update_yaxes(title_text=x_label, row=1, col=2)
    fig.update_yaxes(title_text=y_label, row=1, col=3)
    fig.update_layout(title=title, height=450, template="plotly_white")
    return fig

# ── Plot 1: Thrust bearing ──
fig = plot_group_interactive(
    scenarios=[
        ("normal",              "Healthy",        COLOURS["normal"]),
        ("pump_bearing_wear",   "Bearing wear",   COLOURS["failure"]),
        ("decoy_highload_step", "High-load step", COLOURS["decoy"]),
        ("decoy_highload_ramp", "High-load ramp", "#2196F3"),
    ],
    signal_y="thrust_bearing_K", signal_x="shaft_speed_rads",
    title="Thrust bearing wear vs high-load decoys<br><sub>Failure: Tt higher than expected for its speed | Decoy: Tt tracks speed proportionally</sub>",
    y_label="Thrust bearing Tt (K)", x_label="Shaft speed (rad/s)",
    scatter_label="Speed vs Tt (discriminator)"
)
fig.show()

## Plot 2 — Impeller wear vs back-pressure decoys (observable signals only)

**Both** show flow dropping — but only the failure shows temperature diverging from the healthy baseline at the same shaft speed.

- **Failure:** flow drops + temp rises → hidden `A` degradation (not directly measurable)
- **Hard negative:** flow drops because back-pressure changed — temp stays near healthy

You can't just watch flow. You need to check: "is the temperature rise explained by the operating conditions, or is there unexplained excess heat?"

In [7]:
# ── Plot 2: Impeller wear ──
fig = plot_group_interactive(
    scenarios=[
        ("normal",                   "Healthy",         COLOURS["normal"]),
        ("pump_impeller_wear",       "Impeller wear",   COLOURS["failure"]),
        ("decoy_back_pressure_step", "Back-press step", COLOURS["decoy"]),
        ("decoy_back_pressure_ramp", "Back-press ramp", "#2196F3"),
    ],
    signal_y="fluid_temp_K", signal_x="flow_out_m3s",
    title="Impeller wear vs back-pressure decoys (observable signals only)<br><sub>Failure: temp higher than expected for its flow | Decoy: temp tracks flow proportionally</sub>",
    y_label="Oil temperature To (K)", x_label="Flow out (m\u00b3/s)",
    scatter_label="Flow vs To (discriminator)"
)
fig.show()

## Plot 3 — Radial bearing wear vs high-load decoys

Same logic as thrust bearing: **both** show Tr rising, but the failure has excess heat not explained by load.

- **Failure:** Tr climbs with no speed increase — friction from degraded radial bearing
- **Hard negative:** Higher load → speed ↑ → Tr rises proportionally

In [8]:
# ── Plot 3: Radial bearing ──
fig = plot_group_interactive(
    scenarios=[
        ("normal",                       "Healthy",        COLOURS["normal"]),
        ("pump_radial_wear",             "Radial wear",    COLOURS["failure"]),
        ("decoy_radial_highload_step",   "High-load step", COLOURS["decoy"]),
        ("decoy_radial_highload_ramp",   "High-load ramp", "#2196F3"),
    ],
    signal_y="radial_bearing_K", signal_x="shaft_speed_rads",
    title="Radial bearing wear vs high-load decoys<br><sub>Failure: Tr higher than expected for its speed | Decoy: Tr tracks speed proportionally</sub>",
    y_label="Radial bearing Tr (K)", x_label="Shaft speed (rad/s)",
    scatter_label="Speed vs Tr (discriminator)"
)
fig.show()